# 02 - Memória com LangGraph

Assim como o modelo, o grafo não guarda nada entre uma execução e outra. Cada `invoke` começa do estado que recebeu como entrada.

Aqui fazemos duas coisas:

1. Mostrar um grafo **sem memória**: a segunda execução não sabe o que aconteceu na primeira.
2. Adicionar um **checkpointer** (`InMemorySaver`). Ele salva o estado após cada execução, indexado por `thread_id`, e restaura esse estado na próxima execução da mesma thread.

As configurações de chave e modelo continuam vindo do `.env` na raiz do projeto.

In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(find_dotenv())

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)


def chamar_modelo(estado: MessagesState) -> dict:
    resposta = llm.invoke(estado["messages"])
    return {"messages": [resposta]}


def montar_grafo(checkpointer=None):
    grafo = StateGraph(MessagesState)
    grafo.add_node("modelo", chamar_modelo)
    grafo.add_edge(START, "modelo")
    grafo.add_edge("modelo", END)
    return grafo.compile(checkpointer=checkpointer)

## 1. Grafo sem memória

Compilamos o grafo sem checkpointer e chamamos duas vezes. A segunda entrada contém só a nova pergunta, então o modelo não tem como saber o nome.

In [2]:
app_sem_memoria = montar_grafo()

primeira = app_sem_memoria.invoke({"messages": [("user", "Oi, meu nome é Leonardo e estou estudando LangGraph.")]})
segunda = app_sem_memoria.invoke({"messages": [("user", "Qual é o meu nome e o que eu estou estudando?")]})

display(Markdown(f"**Você:** Oi, meu nome é Leonardo e estou estudando LangGraph.\n\n**Modelo:** {primeira['messages'][-1].content}"))
display(Markdown(f"**Você:** Qual é o meu nome e o que eu estou estudando?\n\n**Modelo:** {segunda['messages'][-1].content}"))

**Você:** Oi, meu nome é Leonardo e estou estudando LangGraph.

**Modelo:** Oi, Leonardo — prazer! Boa sorte nos estudos com LangGraph.

Posso ajudar de várias formas; antes, me diz: você está começando do zero ou já tem algo específico em mãos (ex.: erro, projeto, dúvida conceitual)?  

Resumidamente, o que costuma entrar quando se aprende LangGraph:
- Objetivo: orquestrar modelos de linguagem e ferramentas como um grafo de nós e conexões (fluxos visuais ou programáticos).
- Componentes típicos: nós de LLM (modelos), nós de prompt, nós de ferramentas/funcionalidades (APIs, buscadores, executores), conectores de dados e nós de controle/decisão.
- Vantagens: facilita montar pipelines complexos, reusar subfluxos e visualizar a execução.
- Próximos passos comuns: instalar a biblioteca (se estiver usando a SDK), abrir o editor visual, criar um pequeno fluxo (ex.: prompt → LLM → post-processamento) e testar.

Posso:
- Dar uma explicação mais detalhada dos conceitos;
- Mostrar um exemplo prático (em Python ou no editor visual);
- Ajudar com instalação e dependências;
- Ajudar a depurar um erro ou revisar seu grafo.

Qual você prefere agora?

**Você:** Qual é o meu nome e o que eu estou estudando?

**Modelo:** Não sei — não tenho acesso a informações pessoais ou ao seu perfil. Você ainda não me disse seu nome nem o que está estudando nesta conversa.

Se quiser, diga seu nome e o que estuda (por exemplo: "Meu nome é Ana e estudo engenharia"). Posso então:
- me lembrar disso durante esta conversa;
- ajudar com plano de estudos, resumos, exercícios, correções de texto, etc.;
- sugerir recursos e cronogramas conforme sua área e nível.

## 2. Grafo com checkpointer

Com `InMemorySaver`, cada execução salva o estado completo, incluindo as mensagens anteriores. O `config` com `thread_id` diz qual conversa estamos continuando.

In [3]:
app_com_memoria = montar_grafo(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "aula-1"}}

for pergunta in [
    "Oi, meu nome é Leonardo e estou estudando LangGraph.",
    "Qual é o meu nome e o que eu estou estudando?",
]:
    estado = app_com_memoria.invoke({"messages": [("user", pergunta)]}, config=config)
    display(Markdown(f"**Você:** {pergunta}\n\n**Modelo:** {estado['messages'][-1].content}"))

**Você:** Oi, meu nome é Leonardo e estou estudando LangGraph.

**Modelo:** Oi, Leonardo — prazer! Que bom que você está estudando LangGraph.

Como quer que eu te ajude? Posso:
- Dar uma visão geral rápida do que é LangGraph e quando usar;
- Fazer um tutorial passo a passo com exemplo em Python ou TypeScript;
- Mostrar como integrar com provedores (OpenAI, local LLMs) ou vetores (Chroma, FAISS);
- Ajudar a depurar um erro específico ou revisar um trecho de código;
- Sugerir exercícios e recursos de estudo (docs, repositório, tutoriais).

Me diz seu nível (iniciante / intermediário) e a stack que prefere (Python ou Node/TypeScript) que eu preparo um exemplo ou explicação direcionada. Quer começar com uma visão geral rápida agora?

**Você:** Qual é o meu nome e o que eu estou estudando?

**Modelo:** Seu nome é Leonardo e você está estudando LangGraph. Quer que eu te ajude com algo específico sobre o LangGraph?

Conferindo o estado salvo para a thread `aula-1`:

In [4]:
estado_salvo = app_com_memoria.get_state(config)

for mensagem in estado_salvo.values["messages"]:
    print(f"{mensagem.type}: {mensagem.content}")
    print("-"*10)

human: Oi, meu nome é Leonardo e estou estudando LangGraph.
----------
ai: Oi, Leonardo — prazer! Que bom que você está estudando LangGraph.

Como quer que eu te ajude? Posso:
- Dar uma visão geral rápida do que é LangGraph e quando usar;
- Fazer um tutorial passo a passo com exemplo em Python ou TypeScript;
- Mostrar como integrar com provedores (OpenAI, local LLMs) ou vetores (Chroma, FAISS);
- Ajudar a depurar um erro específico ou revisar um trecho de código;
- Sugerir exercícios e recursos de estudo (docs, repositório, tutoriais).

Me diz seu nível (iniciante / intermediário) e a stack que prefere (Python ou Node/TypeScript) que eu preparo um exemplo ou explicação direcionada. Quer começar com uma visão geral rápida agora?
----------
human: Qual é o meu nome e o que eu estou estudando?
----------
ai: Seu nome é Leonardo e você está estudando LangGraph. Quer que eu te ajude com algo específico sobre o LangGraph?
----------


## 3. Threads separadas

Cada `thread_id` tem o próprio estado. Uma thread nova começa vazia.

In [5]:
estado = app_com_memoria.invoke(
    {"messages": [("user", "Qual é o meu nome?")]},
    config={"configurable": {"thread_id": "aula-2"}},
)

display(Markdown(f"**Você (thread aula-2):** Qual é o meu nome?\n\n**Modelo:** {estado['messages'][-1].content}"))

**Você (thread aula-2):** Qual é o meu nome?

**Modelo:** Eu não sei — não tenho acesso ao seu nome a menos que você me diga. Posso chamar você pelo nome ou por um apelido durante esta conversa. Quer me dizer como prefere ser chamado(a)? (Ex.: nome, apelido, formal/informal)

## Observações

- `InMemorySaver` some quando o processo termina. Para persistir entre execuções, use um checkpointer de banco de dados, como o `SqliteSaver` do pacote `langgraph-checkpoint-sqlite`.
- O histórico inteiro é enviado ao modelo a cada turno, como no notebook `04-Langchain memória` do módulo LLMs.
- Além de mensagens, o checkpointer salva qualquer campo do estado, o que permite guardar outros dados da execução.